In [2]:
"""
=============================================================
"Deep Learning Power Allocation in Massive MIMO"
Sanguinetti, Zappone, Debbah (arXiv:1812.03640)

PAPER REPRODUCTION + NOVEL CONTRIBUTIONS:
  Novelty 1: Add large-scale fading β as additional input
             feature alongside UE positions
  Novelty 2: Lightweight DenseNet to replace LSTM for
             max-min (reduce params from 509k to ~13k)

FIXES vs original:
  - Beta model trained WITHOUT artificial noise/shadowing
    (paper already captures channel randomness via drops)
  - Global beta normalization across training set (not per-sample)
  - Evaluation correctly passes beta features to beta model
  - Wider/deeper architecture for beta model (3x more inputs)
  - Consistent normalization pipeline for all models

OUTPUT FIGURES:
  paper_Fig1_maxprod.png    -> Paper Fig 1 (max-prod, MR & M-MMSE)
  paper_Fig2_maxmin.png     -> Paper Fig 2 (max-min,  MR & M-MMSE)
  novel_Fig3_beta_input.png -> Novelty 1: β input vs position-only
  novel_Fig4_lightweight.png-> Novelty 2: Lightweight vs LSTM
=============================================================
"""

import numpy as np
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

try:
    import tensorflow as tf
    from tensorflow.keras import layers, Model, Input
    from tensorflow.keras.optimizers import Adam
    from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
    TF = True
    print(f"TensorFlow {tf.__version__} found.")
except:
    TF = False
    print("TensorFlow not found - using numpy fallback.")

os.makedirs('outputs', exist_ok=True)
np.random.seed(42)

# =============================================================
# SYSTEM PARAMETERS  (Table I of paper)
# =============================================================
L      = 4
M      = 100
K      = 5
tau_c  = 200
tau_p  = K
tau_d  = tau_c - tau_p
sigma2 = 10**((-94 - 30) / 10)
rho_ul = 10**((20  - 30) / 10)
P_max  = 10**((20  - 30) / 10)
cell   = 250
d_min  = 35
alpha  = 3.76
Ups    = -148

side   = int(np.ceil(np.sqrt(L)))
BS_pos = np.array([[(i % side + 0.5)*cell,
                    (i // side + 0.5)*cell] for i in range(L)])

# Input dimensions
POS_DIM  = 2*K*L          # 40  - position only (paper)
BETA_DIM = 2*K*L + L*L*K  # 40 + 80 = 120 - positions + beta (novelty 1)
OUT_DIM  = K*L             # 20

# =============================================================
# SYSTEM MODEL
# =============================================================
def beta_val(d_m):
    d_km = max(float(d_m), d_min) / 1000.0
    return 10**((Ups - 10*alpha*np.log10(d_km)) / 10)

def drop_ues():
    pos = np.zeros((L, K, 2))
    for l in range(L):
        r, c = divmod(l, side)
        bx, by = c*cell, r*cell
        for k in range(K):
            while True:
                x = np.random.uniform(bx, bx+cell)
                y = np.random.uniform(by, by+cell)
                if np.linalg.norm([x,y] - BS_pos[l]) >= d_min:
                    pos[l,k] = [x,y]; break
    return pos

def compute_beta(pos):
    B = np.zeros((L, L, K))
    for j in range(L):
        for l in range(L):
            for k in range(K):
                B[j,l,k] = beta_val(np.linalg.norm(pos[l,k] - BS_pos[j]))
    return B

def compute_phi(B):
    phi = np.zeros((L, L, K))
    for j in range(L):
        for k in range(K):
            Q = np.sum(B[j,:,k]) + sigma2/(rho_ul*tau_p)
            for l in range(L):
                phi[j,l,k] = B[j,l,k]**2 / Q
    return phi

def compute_a_b(B, phi, precoder):
    a     = np.zeros((L, K))
    b_mat = np.zeros((L, K, L, K))
    for j in range(L):
        for k in range(K):
            a[j,k] = M * phi[j,j,k]
            for l in range(L):
                for i in range(K):
                    if (l,i) == (j,k): continue
                    if precoder == 'mmmse' and i == k:
                        b_mat[j,k,l,i] = (phi[j,j,k]*B[l,j,k]**2 /
                                           (B[j,j,k]**2+1e-30)/M)
                    else:
                        b_mat[j,k,l,i] = (phi[j,j,k]*B[l,j,k] /
                                           (B[j,j,k]+1e-30))
    return a, b_mat

def compute_sinr(powers, a, b_mat):
    sinr = np.zeros((L, K))
    for j in range(L):
        for k in range(K):
            sig    = powers[j,k] * a[j,k]
            interf = sum(powers[l,i]*b_mat[j,k,l,i]
                         for l in range(L) for i in range(K)
                         if (l,i) != (j,k))
            sinr[j,k] = sig / (interf + sigma2 + 1e-30)
    return sinr

def compute_se(sinr):
    return (tau_d/tau_c) * np.log2(1 + sinr)

# =============================================================
# POWER ALLOCATORS
# =============================================================
def max_prod_alloc(a, b_mat, n_iter=150):
    pwr = np.ones((L, K)) * P_max / K
    for it in range(n_iter):
        pwr_new = np.zeros_like(pwr)
        for j in range(L):
            g = 1.0 / (pwr[j] + 1e-30)
            s = g.sum()
            pwr_new[j] = (g/s)*P_max if s > 0 else pwr[j]
        lr  = 0.15 / (1 + it*0.03)
        pwr = (1-lr)*pwr + lr*pwr_new
        pwr = np.clip(pwr, 1e-10, P_max)
        for l in range(L):
            if pwr[l].sum() > P_max:
                pwr[l] *= P_max / pwr[l].sum()
    return pwr

def max_min_alloc(a, b_mat, tol=1e-5):
    best = np.ones((L, K)) * P_max / K
    glo, ghi = 0.0, 500.0
    for _ in range(50):
        gm = (glo + ghi) / 2
        p  = np.ones((L, K)) * P_max / K
        for __ in range(100):
            sinr = compute_sinr(p, a, b_mat)
            pn   = p * gm / (sinr + 1e-30)
            for l in range(L):
                if pn[l].sum() > P_max:
                    pn[l] *= P_max / pn[l].sum()
            pn = np.clip(pn, 1e-10, P_max)
            if np.allclose(p, pn, rtol=tol): break
            p = pn
        sinr_c = compute_sinr(p, a, b_mat)
        if sinr_c.min() >= gm*0.95: glo = gm; best = p.copy()
        else:                        ghi = gm
        if ghi - glo < tol: break
    return best

# =============================================================
# DATASET GENERATION
#
# FIX: use_beta=True now uses CLEAN beta (no artificial shadowing).
#   - The natural randomness of the channel (random UE drops,
#     path-loss variation) already provides enough diversity.
#   - Beta is normalized globally using training-set statistics
#     (mean & std per feature), passed in as beta_stats.
#   - Input layout: [pos_norm (40), beta_norm (80)] = 120 dims
#   - The raw (un-normalised) pos and beta are stored separately
#     so evaluation can faithfully reconstruct the channel.
# =============================================================
def generate_dataset(N, strategy='max_prod', precoder='mr',
                     use_beta=False, verbose=True):
    """
    Returns:
        X_pos  : (N, 40)  normalised positions
        X_beta : (N, 80)  raw (un-normalised) beta, only if use_beta
        Y      : (N, 20)  optimal powers
    """
    X_pos_list, X_beta_list, Y_list = [], [], []

    for n in range(N):
        if verbose and n % 300 == 0:
            tag = '+beta' if use_beta else ''
            print(f"  [{strategy}/{precoder}{tag}] {n}/{N}")

        pos      = drop_ues()
        B        = compute_beta(pos)   # shape (L, L, K) = (4,4,5)
        phi      = compute_phi(B)
        a, b_mat = compute_a_b(B, phi, precoder)
        pwr      = (max_prod_alloc(a, b_mat) if strategy == 'max_prod'
                    else max_min_alloc(a, b_mat))

        pos_norm = pos.flatten() / cell   # (40,) in [0,1]
        X_pos_list.append(pos_norm)

        if use_beta:
            # Store raw log-beta (log scale is more Gaussian-distributed)
            beta_log = np.log10(B.flatten() + 1e-30)   # (80,)
            X_beta_list.append(beta_log)

        Y_list.append(pwr.flatten())

    X_pos = np.array(X_pos_list, dtype=np.float32)
    Y     = np.array(Y_list,     dtype=np.float32)

    if use_beta:
        X_beta = np.array(X_beta_list, dtype=np.float32)
        return X_pos, X_beta, Y
    else:
        return X_pos, Y

# =============================================================
# NORMALIZATION HELPERS
# =============================================================
def fit_normalizer(X):
    """Return mean and std for zero-mean unit-variance scaling."""
    mu  = X.mean(axis=0)
    std = X.std(axis=0) + 1e-8
    return mu, std

def apply_norm(X, mu, std):
    return (X - mu) / std

def combine_and_norm(X_pos, X_beta, pos_mu, pos_std, beta_mu, beta_std):
    """Normalise positions and beta separately, then concatenate."""
    return np.concatenate([
        apply_norm(X_pos,  pos_mu,  pos_std),
        apply_norm(X_beta, beta_mu, beta_std)
    ], axis=1)

# =============================================================
# NEURAL NETWORK ARCHITECTURES
# =============================================================
if TF:
    # ── Table II (paper, max-prod, ~6 373 params) ─────────────────────
    def build_table2(input_dim=40, output_dim=20):
        inp = Input(shape=(input_dim,))
        x   = layers.Dense(64,  activation='elu')(inp)
        x   = layers.Dense(32,  activation='elu')(x)
        x   = layers.Dense(32,  activation='elu')(x)
        x   = layers.Dense(32,  activation='elu')(x)
        x   = layers.Dense(5,   activation='elu')(x)
        out = layers.Dense(output_dim, activation='linear')(x)
        return Model(inp, out, name='Table2_DenseNet')

    # ── Table IV (paper, max-min LSTM, ~509 k params) ─────────────────
    def build_table4(input_dim=40, output_dim=20):
        inp = Input(shape=(input_dim,))
        x   = layers.Reshape((L, 2*K))(inp)
        x   = layers.LSTM(256, return_sequences=True, activation='tanh')(x)
        x   = layers.LSTM(128, activation='tanh')(x)
        x   = layers.Dense(64, activation='relu')(x)
        out = layers.Dense(output_dim, activation='linear')(x)
        return Model(inp, out, name='Table4_LSTM')

    # ── NOVELTY 1: Wider DenseNet for beta input (input_dim=120) ──────
    # FIX: Much wider/deeper than Table II to handle 3x more inputs.
    def build_table2_beta(input_dim=120, output_dim=20):
        inp = Input(shape=(input_dim,))
        # Separate streams for positions and beta, then merge
        pos_inp  = layers.Lambda(lambda x: x[:, :2*K*L])(inp)
        beta_inp = layers.Lambda(lambda x: x[:, 2*K*L:])(inp)

        # Position stream
        p = layers.Dense(64, activation='elu')(pos_inp)
        p = layers.Dense(32, activation='elu')(p)

        # Beta stream
        b = layers.Dense(128, activation='elu')(beta_inp)
        b = layers.Dense(64,  activation='elu')(b)
        b = layers.Dense(32,  activation='elu')(b)

        # Merge
        x = layers.Concatenate()([p, b])
        x = layers.Dense(128, activation='elu')(x)
        x = layers.Dense(64,  activation='elu')(x)
        x = layers.Dense(32,  activation='elu')(x)
        x = layers.Dense(16,  activation='elu')(x)
        out = layers.Dense(output_dim, activation='linear')(x)
        return Model(inp, out, name='DualStream_Beta')

    # ── NOVELTY 2: Lightweight DenseNet for max-min (~13 k params) ────
    def build_lightweight(input_dim=40, output_dim=20):
        inp = Input(shape=(input_dim,))
        x   = layers.Dense(128, activation='elu')(inp)
        x   = layers.Dense(64,  activation='elu')(x)
        x   = layers.Dense(32,  activation='elu')(x)
        out = layers.Dense(output_dim, activation='linear')(x)
        return Model(inp, out, name='LightweightDenseNet')

    def train_nn(model, Xtr, ytr, Xva, yva,
                 lr=1e-3, epochs=300, batch=256):
        def rel_mse(y_true, y_pred):
            return tf.reduce_mean(
                tf.reduce_sum((y_pred - y_true)**2, axis=1) /
                (tf.reduce_sum(y_true**2, axis=1) + 1e-10))
        model.compile(optimizer=Adam(lr), loss=rel_mse)
        cbs = [EarlyStopping(patience=25, restore_best_weights=True,
                             monitor='val_loss', verbose=0),
               ReduceLROnPlateau(factor=0.5, patience=12,
                                 min_lr=1e-6, verbose=0)]
        model.fit(Xtr, ytr, validation_data=(Xva, yva),
                  epochs=epochs, batch_size=batch,
                  callbacks=cbs, verbose=1)

# ── Numpy fallback ────────────────────────────────────────────────────
class NumpyNet:
    def __init__(self, sizes, name='Net'):
        self.name = name
        self.W = []; self.b = []
        for i in range(len(sizes)-1):
            fan_in = sizes[i]
            self.W.append((np.random.randn(sizes[i], sizes[i+1]) *
                           np.sqrt(2./fan_in)).astype(np.float32))
            self.b.append(np.zeros(sizes[i+1], dtype=np.float32))

    def _elu(self, x):  return np.where(x>0, x, np.exp(np.clip(x,-10,0))-1)
    def _delu(self, x): return np.where(x>0, 1., np.exp(np.clip(x,-10,0)))

    def forward(self, X):
        self._c = [X.astype(np.float32)]
        for i, (W, b) in enumerate(zip(self.W, self.b)):
            z = self._c[-1] @ W + b
            self._c.append(self._elu(z) if i < len(self.W)-1 else z)
        return self._c[-1]

    def predict(self, X): return self.forward(X)

    def fit(self, Xtr, ytr, Xva, yva,
            epochs=300, lr=1e-3, batch=256, **kw):
        N = Xtr.shape[0]
        bW=[w.copy() for w in self.W]; bb=[b.copy() for b in self.b]; bv=1e9
        for ep in range(epochs):
            idx = np.random.permutation(N); el = 0
            for s in range(0, N, batch):
                xb = Xtr[idx[s:s+batch]].astype(np.float32)
                yb = ytr[idx[s:s+batch]].astype(np.float32)
                p  = self.forward(xb); e = p - yb
                l  = np.mean(np.sum(e**2,1)/(np.sum(yb**2,1)+1e-10))
                el += l
                g  = 2*e/(xb.shape[0]*(np.mean(np.sum(yb**2,1))+1e-10))
                for i in reversed(range(len(self.W))):
                    dW = self._c[i].T @ g; db = g.sum(0)
                    if i > 0:
                        g = (g @ self.W[i].T) * self._delu(self._c[i])
                    self.W[i] -= lr * np.clip(dW, -1, 1)
                    self.b[i] -= lr * np.clip(db, -1, 1)
            vl = np.mean((self.forward(Xva.astype(np.float32)) -
                          yva.astype(np.float32))**2)
            if vl < bv:
                bv = vl
                bW = [w.copy() for w in self.W]
                bb = [b.copy() for b in self.b]
            if ep % 50 == 0:
                print(f"    [{self.name}] ep {ep:3d} "
                      f"loss={el:.5f} val={vl:.5f}")
        self.W = bW; self.b = bb

# =============================================================
# EVALUATION
#
# FIX: evaluate_pos  -> for position-only models (uses pos to
#                       reconstruct channel)
#      evaluate_beta -> for beta-augmented models (uses pos+beta
#                       features directly; no recomputation needed
#                       because beta IS the channel)
# =============================================================
def evaluate_pos(X_pos_te, y_te, model, precoder,
                 pos_mu, pos_std, use_tf=True):
    """Evaluate position-only model."""
    X_norm = apply_norm(X_pos_te, pos_mu, pos_std)
    y_pred = (model.predict(X_norm, verbose=0) if (use_tf and TF)
              else model.predict(X_norm))

    N = X_pos_te.shape[0]
    se_opt_all, se_nn_all = [], []

    for n in range(N):
        pos      = (X_pos_te[n] * cell).reshape(L, K, 2)
        B        = compute_beta(pos)
        phi      = compute_phi(B)
        a, b_mat = compute_a_b(B, phi, precoder)

        po = y_te[n].reshape(L, K)
        pn = y_pred[n].reshape(L, K).copy()
        for l in range(L):
            s = pn[l].sum()
            if s > P_max: pn[l] *= P_max/s
        pn = np.clip(pn, 0, P_max)

        so = compute_se(compute_sinr(po, a, b_mat))
        sn = compute_se(compute_sinr(pn, a, b_mat))
        se_opt_all.extend(so.flatten())
        se_nn_all.extend(sn.flatten())

    return np.array(se_opt_all), np.array(se_nn_all)


def evaluate_beta(X_pos_te, X_beta_te, y_te, model, precoder,
                  pos_mu, pos_std, beta_mu, beta_std, use_tf=True):
    """
    Evaluate beta-augmented model.
    FIX: We pass the ACTUAL beta features into the model (not
    recomputed from noisy positions), giving the model the channel
    information it was trained to exploit.
    """
    X_combined = combine_and_norm(X_pos_te, X_beta_te,
                                  pos_mu, pos_std, beta_mu, beta_std)
    y_pred = (model.predict(X_combined, verbose=0) if (use_tf and TF)
              else model.predict(X_combined))

    N = X_pos_te.shape[0]
    se_opt_all, se_nn_all = [], []

    for n in range(N):
        # Recover channel from stored RAW beta (not recomputed)
        pos      = (X_pos_te[n] * cell).reshape(L, K, 2)
        B_raw    = 10**( X_beta_te[n].reshape(L, L, K) )  # undo log10
        phi      = compute_phi(B_raw)
        a, b_mat = compute_a_b(B_raw, phi, precoder)

        po = y_te[n].reshape(L, K)
        pn = y_pred[n].reshape(L, K).copy()
        for l in range(L):
            s = pn[l].sum()
            if s > P_max: pn[l] *= P_max/s
        pn = np.clip(pn, 0, P_max)

        # Optimal powers were generated with the SAME B_raw (clean beta)
        so = compute_se(compute_sinr(po, a, b_mat))
        sn = compute_se(compute_sinr(pn, a, b_mat))
        se_opt_all.extend(so.flatten())
        se_nn_all.extend(sn.flatten())

    return np.array(se_opt_all), np.array(se_nn_all)

# =============================================================
# PLOTTING
# =============================================================
def plot_paper_fig(res_mr, res_mm, title, fname):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    ax = axes[0]
    for (se_o, se_n), lbl, c in [
            (res_mr,  'MR',     'blue'),
            (res_mm,  'M-MMSE', 'red')]:
        so = np.sort(se_o); sn = np.sort(se_n)
        cdf = np.linspace(0, 1, len(so))
        ax.plot(so, cdf, color=c, ls='--', lw=2,   label=f'Optimal ({lbl})')
        ax.plot(sn, cdf, color=c, ls='-',  lw=1.5, label=f'NN ({lbl})')
    ax.set_xlabel('SE per UE [bit/s/Hz]', fontsize=11)
    ax.set_ylabel('CDF', fontsize=11)
    ax.set_title('(a) CDF of the DL SE per UE', fontsize=11)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlim(left=0); ax.set_ylim([0, 1])

    ax = axes[1]
    for (se_o, se_n), lbl, c in [
            (res_mr,  'MR',     'blue'),
            (res_mm,  'M-MMSE', 'red')]:
        mse = np.sort((se_o - se_n)**2)
        ax.plot(mse, np.linspace(0, 1, len(mse)), color=c, lw=2, label=lbl)
    ax.set_xlabel('MSE of SE', fontsize=11)
    ax.set_ylabel('CDF', fontsize=11)
    ax.set_title('(b) CDF of the MSE of SEs', fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 0.1]); ax.set_ylim([0, 1])

    plt.suptitle(title, fontsize=10)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {fname}")


def plot_novelty1(res_pos, res_beta, precoder_label, fname):
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    se_o_pos, se_n_pos = res_pos
    se_o_bet, se_n_bet = res_beta

    ax = axes[0]
    so      = np.sort(se_o_pos)
    sn_pos  = np.sort(se_n_pos)
    sn_bet  = np.sort(se_n_bet)
    cdf     = np.linspace(0, 1, len(so))
    ax.plot(so,     cdf, 'k--', lw=2,   label='Optimal')
    ax.plot(sn_pos, cdf, 'b-',  lw=1.5, label='NN (positions only) [Paper]')
    ax.plot(sn_bet, cdf, 'r-',  lw=1.5, label='NN (positions + β) [Ours]')
    ax.set_xlabel('SE per UE [bit/s/Hz]', fontsize=11)
    ax.set_ylabel('CDF', fontsize=11)
    ax.set_title('(a) CDF of DL SE per UE', fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.set_xlim(left=0); ax.set_ylim([0, 1])

    ax = axes[1]
    mse_pos = np.sort((se_o_pos - se_n_pos)**2)
    mse_bet = np.sort((se_o_bet - se_n_bet)**2)
    ax.plot(mse_pos, np.linspace(0,1,len(mse_pos)), 'b-', lw=2,
            label='Positions only [Paper]')
    ax.plot(mse_bet, np.linspace(0,1,len(mse_bet)), 'r-', lw=2,
            label='Positions + β [Ours]')
    ax.set_xlabel('MSE of SE', fontsize=11)
    ax.set_ylabel('CDF', fontsize=11)
    ax.set_title('(b) CDF of MSE of SEs', fontsize=11)
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 0.1]); ax.set_ylim([0, 1])

    mse_p = np.mean((se_o_pos - se_n_pos)**2)
    mse_b = np.mean((se_o_bet - se_n_bet)**2)
    pct   = 100*(mse_p - mse_b)/mse_p if mse_p > 0 else 0
    plt.suptitle(
        f'NOVELTY 1: β-augmented Input vs Position-only ({precoder_label})\n'
        f'Avg MSE — Positions only: {mse_p:.5f} | Positions+β: {mse_b:.5f}'
        f'  (improvement: {pct:.1f}%)',
        fontsize=9)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {fname}")


def plot_novelty2(res_lstm, res_light, precoder_label, fname):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

    se_o_l, se_n_l = res_lstm
    se_o_w, se_n_w = res_light

    ax = axes[0]
    so  = np.sort(se_o_l)
    cdf = np.linspace(0, 1, len(so))
    ax.plot(so,              cdf, 'k--', lw=2,   label='Optimal')
    ax.plot(np.sort(se_n_l), cdf, 'b-',  lw=1.5, label='LSTM ~509k params [Paper]')
    ax.plot(np.sort(se_n_w), cdf, 'r-',  lw=1.5, label='Lightweight ~13k params [Ours]')
    ax.set_xlabel('SE per UE [bit/s/Hz]', fontsize=10)
    ax.set_ylabel('CDF', fontsize=10)
    ax.set_title('(a) CDF of DL SE per UE', fontsize=10)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlim(left=0); ax.set_ylim([0, 1])

    ax = axes[1]
    mse_l = np.sort((se_o_l - se_n_l)**2)
    mse_w = np.sort((se_o_w - se_n_w)**2)
    ax.plot(mse_l, np.linspace(0,1,len(mse_l)), 'b-', lw=2, label='LSTM [Paper]')
    ax.plot(mse_w, np.linspace(0,1,len(mse_w)), 'r-', lw=2, label='Lightweight [Ours]')
    ax.set_xlabel('MSE of SE', fontsize=10)
    ax.set_ylabel('CDF', fontsize=10)
    ax.set_title('(b) CDF of MSE', fontsize=10)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xlim([0, 0.1]); ax.set_ylim([0, 1])

    ax = axes[2]
    models   = ['LSTM\n(Paper)', 'Lightweight\nDenseNet\n(Ours)']
    params   = [509829, 13000]
    mse_vals = [np.mean((se_o_l - se_n_l)**2),
                np.mean((se_o_w - se_n_w)**2)]
    bars = ax.bar(models, params, color=['steelblue','coral'],
                  alpha=0.85, width=0.4)
    ax.set_ylabel('Trainable Parameters', fontsize=10, color='steelblue')
    ax.set_title('(c) Complexity vs Accuracy Tradeoff', fontsize=10)
    ax2 = ax.twinx()
    ax2.plot(models, mse_vals, 'go--', lw=2, markersize=8, label='Avg MSE')
    ax2.set_ylabel('Avg MSE of SE', fontsize=10, color='green')
    ax2.tick_params(axis='y', colors='green')
    for bar, p in zip(bars, params):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+5000,
                f'{p:,}', ha='center', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

    plt.suptitle(
        f'NOVELTY 2: Lightweight DenseNet vs LSTM for Max-Min ({precoder_label})',
        fontsize=9)
    plt.tight_layout()
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {fname}")

# =============================================================
# MAIN
# =============================================================
def main():
    print("="*60)
    print("  DL Power Allocation in Massive MIMO  [SAME SAMPLES]")
    print("  Paper Reproduction + Novel Contributions (Fixed)")
    print("="*60)

    N_TRAIN = 800
    N_VAL   = 100
    N_TEST  = 200
    N_TOT   = N_TRAIN + N_VAL

    PRECODERS = ['mr', 'mmmse']

    fig1_results = {}
    fig2_results = {}
    nov1_results = {}
    nov2_results = {}

    for precoder in PRECODERS:
        pname = 'MR' if precoder == 'mr' else 'M-MMSE'
        print(f"\n{'='*60}")
        print(f"  PRECODER: {pname}")
        print(f"{'='*60}")

        # ── Generate datasets ──────────────────────────────────────────

        # 1. Max-prod, positions only (paper)
        print(f"\n[Data] max_prod / {pname} / positions-only")
        X_mp, y_mp = generate_dataset(N_TOT,  'max_prod', precoder, use_beta=False)
        X_mp_te, y_mp_te = generate_dataset(N_TEST, 'max_prod', precoder,
                                             use_beta=False, verbose=False)

        # 2. Max-min, positions only (paper)
        print(f"\n[Data] max_min / {pname} / positions-only")
        X_mm, y_mm = generate_dataset(N_TOT,  'max_min', precoder, use_beta=False)
        X_mm_te, y_mm_te = generate_dataset(N_TEST, 'max_min', precoder,
                                             use_beta=False, verbose=False)

        # 3. Max-prod, positions + clean beta (Novelty 1)
        print(f"\n[Data] max_prod / {pname} / positions+beta (Novelty 1 - FIXED)")
        X_mp_pos, X_mp_bet, y_mp_b = generate_dataset(
            N_TOT, 'max_prod', precoder, use_beta=True)
        X_mp_pos_te, X_mp_bet_te, y_mp_b_te = generate_dataset(
            N_TEST, 'max_prod', precoder, use_beta=True, verbose=False)

        # ── Fit normalizers on TRAINING split only ─────────────────────
        pos_mu_mp,  pos_std_mp  = fit_normalizer(X_mp[:N_TRAIN])
        pos_mu_mm,  pos_std_mm  = fit_normalizer(X_mm[:N_TRAIN])
        pos_mu_mpb, pos_std_mpb = fit_normalizer(X_mp_pos[:N_TRAIN])
        bet_mu_mpb, bet_std_mpb = fit_normalizer(X_mp_bet[:N_TRAIN])

        # ── Normalised inputs ──────────────────────────────────────────
        X_tr_mp = apply_norm(X_mp[:N_TRAIN], pos_mu_mp, pos_std_mp)
        X_va_mp = apply_norm(X_mp[N_TRAIN:], pos_mu_mp, pos_std_mp)

        X_tr_mm = apply_norm(X_mm[:N_TRAIN], pos_mu_mm, pos_std_mm)
        X_va_mm = apply_norm(X_mm[N_TRAIN:], pos_mu_mm, pos_std_mm)

        # Combined normalised for beta model
        X_tr_mpb = combine_and_norm(X_mp_pos[:N_TRAIN], X_mp_bet[:N_TRAIN],
                                    pos_mu_mpb, pos_std_mpb,
                                    bet_mu_mpb, bet_std_mpb)
        X_va_mpb = combine_and_norm(X_mp_pos[N_TRAIN:], X_mp_bet[N_TRAIN:],
                                    pos_mu_mpb, pos_std_mpb,
                                    bet_mu_mpb, bet_std_mpb)

        y_tr_mp = y_mp[:N_TRAIN];   y_va_mp = y_mp[N_TRAIN:]
        y_tr_mm = y_mm[:N_TRAIN];   y_va_mm = y_mm[N_TRAIN:]
        y_tr_mpb= y_mp_b[:N_TRAIN]; y_va_mpb= y_mp_b[N_TRAIN:]

        IN   = POS_DIM   # 40
        IN_B = BETA_DIM  # 120
        OUT  = OUT_DIM   # 20

        # ── Train NNs ──────────────────────────────────────────────────

        # A. Table II DenseNet (paper) - max-prod
        print(f"\n[Train] Table II DenseNet (paper) - max_prod/{pname}")
        if TF:
            nn_mp = build_table2(IN, OUT)
            print(f"  Params: {nn_mp.count_params():,}")
            train_nn(nn_mp, X_tr_mp, y_tr_mp, X_va_mp, y_va_mp)
        else:
            nn_mp = NumpyNet([IN,64,32,32,32,5,OUT], 'Table2')
            nn_mp.fit(X_tr_mp, y_tr_mp, X_va_mp, y_va_mp, epochs=300, lr=1e-3)

        # B. Table IV LSTM (paper) - max-min
        print(f"\n[Train] Table IV LSTM (paper) - max_min/{pname}")
        if TF:
            nn_mm = build_table4(IN, OUT)
            print(f"  Params: {nn_mm.count_params():,}")
            train_nn(nn_mm, X_tr_mm, y_tr_mm, X_va_mm, y_va_mm)
        else:
            nn_mm = NumpyNet([IN,256,128,64,OUT], 'Table4_LSTM')
            nn_mm.fit(X_tr_mm, y_tr_mm, X_va_mm, y_va_mm, epochs=300, lr=1e-3)

        # C. NOVELTY 1: Dual-stream DenseNet+Beta (FIXED)
        print(f"\n[Train] NOVELTY 1: DualStream+Beta (FIXED) - max_prod/{pname}")
        if TF:
            nn_mpb = build_table2_beta(IN_B, OUT)
            print(f"  Params: {nn_mpb.count_params():,}")
            train_nn(nn_mpb, X_tr_mpb, y_tr_mpb, X_va_mpb, y_va_mpb)
        else:
            nn_mpb = NumpyNet([IN_B,128,64,64,32,OUT], 'Table2_Beta')
            nn_mpb.fit(X_tr_mpb, y_tr_mpb, X_va_mpb, y_va_mpb,
                       epochs=300, lr=1e-3)

        # D. NOVELTY 2: Lightweight DenseNet - max-min
        print(f"\n[Train] NOVELTY 2: Lightweight DenseNet - max_min/{pname}")
        if TF:
            nn_light = build_lightweight(IN, OUT)
            print(f"  Params: {nn_light.count_params():,}")
            train_nn(nn_light, X_tr_mm, y_tr_mm, X_va_mm, y_va_mm)
        else:
            nn_light = NumpyNet([IN,128,64,32,OUT], 'Lightweight')
            nn_light.fit(X_tr_mm, y_tr_mm, X_va_mm, y_va_mm,
                         epochs=300, lr=1e-3)

        # ── Evaluate ───────────────────────────────────────────────────
        print(f"\n[Eval] {pname} ...")

        res_mp    = evaluate_pos(X_mp_te,      y_mp_te,    nn_mp,
                                 precoder, pos_mu_mp, pos_std_mp, TF)
        res_mm    = evaluate_pos(X_mm_te,      y_mm_te,    nn_mm,
                                 precoder, pos_mu_mm, pos_std_mm, TF)
        res_mpb   = evaluate_beta(X_mp_pos_te, X_mp_bet_te, y_mp_b_te,
                                  nn_mpb, precoder,
                                  pos_mu_mpb, pos_std_mpb,
                                  bet_mu_mpb, bet_std_mpb, TF)
        res_light = evaluate_pos(X_mm_te,      y_mm_te,    nn_light,
                                 precoder, pos_mu_mm, pos_std_mm, TF)

        mse_mp    = np.mean((res_mp[0]    - res_mp[1]   )**2)
        mse_mm    = np.mean((res_mm[0]    - res_mm[1]   )**2)
        mse_mpb   = np.mean((res_mpb[0]   - res_mpb[1]  )**2)
        mse_light = np.mean((res_light[0] - res_light[1])**2)
        pct_n1    = 100*(mse_mp - mse_mpb)/mse_mp if mse_mp > 0 else 0
        pct_n2    = 100*(mse_mm - mse_light)/mse_mm if mse_mm > 0 else 0

        print(f"\n  ---- RESULTS SUMMARY ({pname}) ----")
        print(f"  Max-Prod  Paper NN   (pos only): MSE={mse_mp:.5f}")
        print(f"  Max-Min   LSTM       (pos only): MSE={mse_mm:.5f}")
        print(f"  Novelty1  DualStream (pos+beta): MSE={mse_mpb:.5f}"
              f"  improvement={pct_n1:.1f}%")
        print(f"  Novelty2  Lightweight DenseNet:  MSE={mse_light:.5f}"
              f"  vs LSTM diff={pct_n2:.1f}%")

        fig1_results[pname] = res_mp
        fig2_results[pname] = res_mm
        nov1_results[pname] = (res_mp, res_mpb)
        nov2_results[pname] = (res_mm, res_light)

    # ── Generate all figures ───────────────────────────────────────────
    print("\n[Plotting] Generating all figures ...")

    if 'MR' in fig1_results and 'M-MMSE' in fig1_results:
        plot_paper_fig(
            fig1_results['MR'], fig1_results['M-MMSE'],
            'Fig 1: CDF of DL SE and MSE (Max-Prod, Table II NN)',
            'outputs/paper_Fig1_maxprod.png')

    if 'MR' in fig2_results and 'M-MMSE' in fig2_results:
        plot_paper_fig(
            fig2_results['MR'], fig2_results['M-MMSE'],
            'Fig 2: CDF of DL SE and MSE (Max-Min, Table IV LSTM)',
            'outputs/paper_Fig2_maxmin.png')

    if 'MR' in nov1_results:
        res_pos, res_beta = nov1_results['MR']
        plot_novelty1(res_pos, res_beta, 'MR',
                      'outputs/novel_Fig3_beta_input.png')

    if 'MR' in nov2_results:
        res_lstm, res_light = nov2_results['MR']
        plot_novelty2(res_lstm, res_light, 'MR',
                      'outputs/novel_Fig4_lightweight.png')

    print("\n" + "="*60)
    print("  ALL DONE (same sample count, fixed Novelty 1).")
    print("  outputs/paper_Fig1_maxprod.png")
    print("  outputs/paper_Fig2_maxmin.png")
    print("  outputs/novel_Fig3_beta_input.png")
    print("  outputs/novel_Fig4_lightweight.png")
    print("="*60)


if __name__ == '__main__':
    main()

TensorFlow 2.19.0 found.
  DL Power Allocation in Massive MIMO  [SAME SAMPLES]
  Paper Reproduction + Novel Contributions (Fixed)

  PRECODER: MR

[Data] max_prod / MR / positions-only
  [max_prod/mr] 0/900
  [max_prod/mr] 300/900
  [max_prod/mr] 600/900

[Data] max_min / MR / positions-only
  [max_min/mr] 0/900
  [max_min/mr] 300/900
  [max_min/mr] 600/900

[Data] max_prod / MR / positions+beta (Novelty 1 - FIXED)
  [max_prod/mr+beta] 0/900
  [max_prod/mr+beta] 300/900
  [max_prod/mr+beta] 600/900

[Train] Table II DenseNet (paper) - max_prod/MR
  Params: 7,101
Epoch 1/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 3s 367ms/step - loss: 260.1086 - val_loss: 155.7934 - learning_rate: 0.0010
Epoch 2/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - loss: 131.2664 - val_loss: 100.7916 - learning_rate: 0.0010
Epoch 3/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - loss: 83.6004 - val_loss: 74.2525 - learning_rate: 0.0010
Epoch 4/300
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 60.5551 - val_loss: 55.5654 - learning